# Extra - Multimodal fusion: images and clinical data together

**Do this notebook only if you have finished notebooks 03 and 04.**

A doctor does not decide from the image alone. The doctor also knows the age of the patient, the smoking
history, how well the patient is doing in daily life. The question of this notebook is:

**Does a model that sees both the image and the clinical data predict better than a model that sees only one?**

**Warning: the clinical data and the outcome of this notebook are invented.** The images are real (the
nodule patches of notebook 04). The age, the smoking history and the survival of each patient are generated
by a formula. Nothing here has a medical meaning: the point is the method. At the end the formula is revealed.

| Part | Steps | What you do |
|---|---|---|
| A | STEP 1 to 3 | patients, clinical table, training and test patients |
| B | STEP 4 to 5 | one model on the clinical data, one model on measurements of the image |
| C | STEP 6 to 7 | early fusion and late fusion |
| D | STEP 8 to 9 | image embeddings, and a network that learns from both |
| E | STEP 10 | compare, and open the box |

As before: replace every `...`, use the variable names of the instructions, and compare your result with
"You should see". No GPU is needed: everything runs on the CPU in a few minutes.

# STEP 0: Setup

Run the cells.

In [ ]:
!pip install -q monai

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from monai.networks.nets import DenseNet121

In [ ]:
DATA_DIR = Path("/kaggle/input/luna16-lung-slices") / "nodules"

np.random.seed(0)
torch.manual_seed(0)

# STEP 1: The patients

In notebook 04 a patient could have several nodules. Here we want **one row per patient**: we keep the
largest nodule of each patient.

1. Read `metadata.csv` into `metadata`.
2. Keep the largest nodule of each patient in `patients`.
3. Print how many patients there are.

In [ ]:
# 1. The table of the nodules
metadata = pd.read_csv(DATA_DIR / "metadata.csv")

# 2. The largest nodule of each patient
patients = metadata.sort_values("diameter_mm", ascending=False).drop_duplicates("case_id").sort_values("case_id").reset_index(drop=True)

# 3. How many patients
print(len(patients))

**You should see** `600`.

The next cell loads the 600 images and masks into two arrays, `images` and `masks`, and shows six patches
with the border of the mask. Run the cell.

In [ ]:
images = np.stack([nib.load(DATA_DIR / "images" / f"{n}.nii.gz").get_fdata() for n in patients["nodule_id"]])
masks = np.stack([nib.load(DATA_DIR / "labels" / f"{n}.nii.gz").get_fdata() for n in patients["nodule_id"]]) > 0.5
print(images.shape, masks.shape)

plt.figure(figsize=(16, 3))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.title(f"{patients['diameter_mm'][i]} mm")
    plt.imshow(images[i].T, cmap="gray", vmin=-1000, vmax=400)
    plt.contour(masks[i].T, levels=[0.5], colors="red")
    plt.axis("off")
plt.show()

**You should see** `(600, 64, 64) (600, 64, 64)` and six patches with a red border around the nodule. The masks
are the outlines drawn by the radiologists of the LIDC-IDRI project.

# STEP 2: The clinical table

Real clinical data cannot be shared in a course, so we generate it. The cell below defines the function
`make_clinical_table` and creates `clinical`, one row per patient. **Run it without reading it.** We open
it at the end, in STEP 10.

In [ ]:
def make_clinical_table(patients, images, masks, seed=0):
    rng = np.random.default_rng(seed)
    n = len(patients)
    z = lambda v: (v - v.mean()) / v.std()
    area = masks.sum(axis=(1, 2)) * patients["spacing_x"].values * patients["spacing_y"].values
    diameter = 2 * np.sqrt(area / np.pi)
    mean_hu = np.array([img[m].mean() for img, m in zip(images, masks)])
    age = np.clip(rng.normal(66, 8, n), 45, 88).round()
    sex = rng.choice(["F", "M"], n)
    pack_years = np.clip(rng.gamma(2.5, 14, n), 0, 120).round()
    performance_status = rng.choice([0, 1, 2], n, p=[0.5, 0.35, 0.15])
    risk = (-1.0 + 1.0 * z(np.log(diameter)) + 0.4 * z(mean_hu) + 0.6 * z(age)
            + 0.8 * (performance_status - 0.65) + 0.4 * z(pack_years))
    alive_1y = (rng.random(n) > 1 / (1 + np.exp(-risk))).astype(int)
    return pd.DataFrame({"case_id": patients["case_id"].values, "age": age, "sex": sex, "pack_years": pack_years,
                         "performance_status": performance_status, "alive_1y": alive_1y})


clinical = make_clinical_table(patients, images, masks, seed=0)

The columns of `clinical`:

| Column | Meaning |
|---|---|
| `case_id` | the patient |
| `age` | age in years |
| `sex` | F or M |
| `pack_years` | smoking history: packs per day times years of smoking |
| `performance_status` | 0 = fully active, 1 = some limits, 2 = in bed part of the day |
| `alive_1y` | **what we want to predict**: 1 = alive one year later, 0 = not |

1. Print the first rows of `clinical`.
2. Print the fraction of patients alive after one year.
3. Print the fraction alive for each value of `performance_status`.

In [ ]:
# 1. The first rows
print(clinical.head())

# 2. Fraction alive
print(clinical["alive_1y"].mean())

# 3. Fraction alive by performance status
print(clinical.groupby("performance_status")["alive_1y"].mean())

**You should see** the first five rows of the table, a fraction alive of about 0.70, and the fraction alive by performance
status: about 0.81 for status 0, 0.64 for status 1, and 0.44 for status 2.

The patients who are more active live longer, as one would expect.

# STEP 3: Training and test patients

As always, the model learns on the training patients and is judged on the test patients.

1. Put the column `alive_1y` as a numpy array in `y`.
2. Split the numbers `0 ... 600` into `train_idx` and `test_idx`: 30% test, `stratify=y` (the same fraction of
   alive patients in both), `random_state=42`.

In [ ]:
# 1. What we predict
y = clinical["alive_1y"].values

# 2. The split
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=42)

The next cell defines `test_auc(features)`. It is used in all the following steps. It trains a logistic
regression on the training patients and returns the AUC on the test patients, and the probabilities of the
test patients. Run the cell.

The AUC is the area under the ROC curve: 0.5 is a coin toss, 1.0 is a perfect model.
`results` is a dictionary that collects the AUC of every model.

In [ ]:
def test_auc(features):
    # trains a logistic regression on the training patients, returns the AUC and the probabilities on the test patients
    X = np.asarray(features, dtype=float)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(X[train_idx], y[train_idx])
    probabilities = model.predict_proba(X[test_idx])[:, 1]
    return roc_auc_score(y[test_idx], probabilities), probabilities


results = {}

# STEP 4: Model 1, clinical data only

1. Build the DataFrame `X_clinical` with four columns: `age`, `pack_years`, `performance_status`, and `male`
   (1 if `sex` is "M", else 0).
2. Call `test_auc` on it. Keep the two results in `auc` and `p_clinical`.
3. Save the AUC in `results["clinical only"]` and print it.

In [ ]:
# 1. The clinical columns
X_clinical = clinical[["age", "pack_years", "performance_status"]].copy()
X_clinical["male"] = (clinical["sex"] == "M").astype(int)

# 2. Train and test
auc, p_clinical = test_auc(X_clinical)

# 3. Save the AUC
results["clinical only"] = auc
print(auc)

**You should see** an AUC of about 0.76. Better than a coin toss, far from perfect.

# STEP 5: Numbers from the image

A logistic regression cannot read pixels. But the mask tells us where the nodule is, so we can turn the nodule
into a few numbers: its size, its brightness, how much the brightness varies. Measuring a nodule like this is
the idea of **radiomics** (shape, intensity, texture).

First one patient, number 0.

1. `area_mm2`: the number of pixels of the mask, times `spacing_x` and `spacing_y` of the patient (in mm).
2. `diameter_mm`: the diameter of the circle with the same area (area = pi * (diameter / 2) ** 2).
3. `values`: the Hounsfield values of the pixels inside the mask. Then print their mean, standard deviation
   and maximum.
4. Print the `diameter_mm` of the metadata and compare.

Note: a few patches hold two nodules. The mask has both, and the area sums them.

In [ ]:
# 1. Area in mm2
area_mm2 = masks[0].sum() * patients["spacing_x"][0] * patients["spacing_y"][0]

# 2. Diameter of the circle with the same area
diameter_mm = 2 * np.sqrt(area_mm2 / np.pi)

# 3. Values inside the mask
values = images[0][masks[0]]
print(area_mm2, diameter_mm, values.mean(), values.std(), values.max())

# 4. The diameter of the metadata
print(patients["diameter_mm"][0])

**You should see** the area (about 24 mm2), the diameter (about 5.5 mm), a mean of about -164 HU, a standard deviation of
about 140 HU and a maximum of 28 HU. The diameter from the mask (5.5 mm) is not the one of the metadata
(6.44 mm). The two do not have to agree: the metadata value is an average over several radiologists and
slices, our measurement is one outline on one slice.

Now all the patients. The loop goes through the patients and builds one row of five numbers for each.

1. `area_mm2`: as before.
2. `values`: as before.
3. The row: area, diameter, mean, standard deviation and maximum.
4. Then the table `X_measurements` is built for you. Save the AUC in `results["image measurements only"]`
   and keep the probabilities in `p_measurements`.

In [ ]:
rows = []
for image, mask, sx, sy in zip(images, masks, patients["spacing_x"], patients["spacing_y"]):
    # 1. Area
    area_mm2 = mask.sum() * sx * sy
    # 2. Values inside the mask
    values = image[mask]
    # 3. The row
    rows.append([area_mm2, 2 * np.sqrt(area_mm2 / np.pi), values.mean(), values.std(), values.max()])

X_measurements = pd.DataFrame(rows, columns=["area_mm2", "diameter_mm", "mean_hu", "std_hu", "max_hu"])

# 4. Train, test and save
auc, p_measurements = test_auc(X_measurements)
results["image measurements only"] = auc
print(auc)

**You should see** an AUC of about 0.74. Five numbers from the image do as well as the four clinical
columns (0.76): the difference is noise.

Note: these are the reference masks of the dataset. In a real project you could use the masks predicted by
your model of notebook 04.

# STEP 6: Early fusion

**Early fusion**: put the columns of the two sources side by side in one table, and train **one** model on it.

1. Build `X_early` with `pd.concat([X_measurements, X_clinical], axis=1)`.
2. Test it and save the AUC in `results["early fusion"]`.

In [ ]:
# 1. The columns side by side
X_early = pd.concat([X_measurements, X_clinical], axis=1)

# 2. Train, test and save
auc, p_early = test_auc(X_early)
results["early fusion"] = auc
print(auc)

**You should see** an AUC of about 0.87, clearly higher than both models before (0.76 and 0.74).

Think: why is it better than both? What does the image know that the table does not, and the other way round?

# STEP 7: Late fusion

**Late fusion**: two models, each one on its own data. Then we combine their answers. You already have the
probabilities of the two models: `p_clinical` and `p_measurements`.

1. `p_late` is the average of the two probabilities.
2. Compute the AUC of `p_late` on the test patients with `roc_auc_score(y[test_idx], p_late)`, and save it
   in `results["late fusion"]`.

In [ ]:
# 1. The average
p_late = (p_clinical + p_measurements) / 2

# 2. The AUC
results["late fusion"] = roc_auc_score(y[test_idx], p_late)
print(results["late fusion"])

**You should see** an AUC of about 0.86, the same as early fusion.

Late fusion is handy when one source is missing for some patients (use the model that has data), or when the two
models were built by different teams and nobody can retrain them together.

# STEP 8: Embeddings from a pretrained network

In STEP 5 we chose the measurements by hand. Another way: pass the patch through a network that was trained
on a million everyday photographs (ImageNet), and keep the numbers of its last layer. Those 1024 numbers are
the **embedding** of the patch. We do not train anything: we only use the network as a measuring tool.

The next cell builds the network (DenseNet121 of MONAI, with its pretrained weights), prepares the patches
(window from -1000 / 400 HU to 0 / 1, three equal channels because the network expects colour, 224 x 224
pixels, the mean and standard deviation of ImageNet) and computes `embeddings`. Run the cell. It takes about
a minute. The weights are downloaded the first time, so Internet must be on in the notebook settings.

In [ ]:
net = DenseNet121(spatial_dims=2, in_channels=3, out_channels=1000, pretrained=True).eval()

x = torch.tensor(np.clip((images + 1000) / 1400, 0, 1), dtype=torch.float32)[:, None]
x = x.repeat(1, 3, 1, 1)
x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
x = (x - torch.tensor([0.485, 0.456, 0.406])[None, :, None, None]) / torch.tensor([0.229, 0.224, 0.225])[None, :, None, None]

with torch.no_grad():
    embeddings = []
    for start in range(0, len(x), 50):
        features = F.relu(net.features(x[start:start + 50]))
        embeddings.append(F.adaptive_avg_pool2d(features, 1).flatten(1))
    embeddings = torch.cat(embeddings).numpy()

Now you use it.

1. Print the shape of `embeddings`.
2. 1024 numbers are many for 420 training patients. Reduce them to 10 with a PCA fitted **on the training
   patients only**: `pca`, then `X_embeddings`.
3. The cell after this one draws the first two components, coloured by the diameter of the nodule. Run it.
4. Test `X_embeddings` alone, and `X_embeddings` together with `X_clinical` (concatenate them as in STEP 6).
   Save the AUCs in `results["embeddings only"]` and `results["early fusion with embeddings"]`.

In [ ]:
# 1. Shape
print(embeddings.shape)

# 2. PCA fitted on the training patients only
pca = PCA(n_components=10, random_state=0).fit(embeddings[train_idx])
X_embeddings = pca.transform(embeddings)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_embeddings[:, 0], X_embeddings[:, 1], c=patients["diameter_mm"], cmap="viridis", s=12)
plt.colorbar(label="diameter of the nodule (mm)")
plt.xlabel("component 1")
plt.ylabel("component 2")
plt.show()

In [ ]:
# 4. Embeddings alone, and embeddings with the clinical data
auc, p_embeddings = test_auc(X_embeddings)
results["embeddings only"] = auc
print(auc)

auc, p_early_embeddings = test_auc(np.concatenate([X_embeddings, X_clinical.values], axis=1))
results["early fusion with embeddings"] = auc
print(auc)

**You should see** the shape `(600, 1024)`. In the plot the colours are mixed: only a weak pattern, with the large nodules a little more often at the top. The AUCs: embeddings only about 0.62,
embeddings with clinical data about 0.79.

The embeddings are the weakest source: below the clinical data (0.76) and below the five
measurements of STEP 5 (0.74). With the clinical data they reach 0.79, well below early fusion with the
measurements (0.87). This is a real
result, and there are three reasons. The network never saw a CT scan. Nobody told it where the nodule is: it
looks at the whole patch. And 420 patients are few for 1024 numbers, even after the PCA.

# STEP 9: Intermediate fusion

In early fusion the image numbers are fixed before the model starts. In **intermediate fusion** one network
takes the image and the table together, and learns the image numbers at the same time as the final answer:

- the image goes through three small convolution layers and becomes 16 numbers;
- the four clinical columns go through one linear layer and become 8 numbers;
- the 16 and the 8 numbers are joined, and one linear layer gives the answer.

The code is written. You fill two blanks: where the two sets of numbers are joined (first cell) and the call of
the network in the training loop (second cell). Read the cells first.

The next cell prepares the data as tensors: the image from 0 to 1, the clinical columns standardised with the
mean and standard deviation of the training patients. Run it.

In [ ]:
image_tensor = torch.tensor(np.clip((images + 1000) / 1400, 0, 1), dtype=torch.float32)[:, None]
table = X_clinical.values.astype(float)
table = (table - table[train_idx].mean(axis=0)) / table[train_idx].std(axis=0)
table_tensor = torch.tensor(table, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

1. In `forward`, join `image_features` and `table_features` side by side into `both`.

In [ ]:
class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_branch = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.table_branch = nn.Sequential(nn.Linear(4, 8), nn.ReLU())
        self.head = nn.Linear(16 + 8, 1)

    def forward(self, image, table):
        image_features = self.image_branch(image)      # 16 numbers
        table_features = self.table_branch(table)      # 8 numbers
        # 1. Join them
        both = torch.cat([image_features, table_features], dim=1)
        return self.head(both).squeeze(1)

2. In the training loop, give the network the images and the table rows of the batch, and keep the answer in
   `logits`. Then run the cell: 40 epochs, about a minute. The loss should go down.

In [ ]:
torch.manual_seed(0)
net = FusionNet()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_function = nn.BCEWithLogitsLoss()
train_tensor = torch.tensor(train_idx)

for epoch in range(40):
    net.train()
    epoch_loss = 0
    order = train_tensor[torch.randperm(len(train_tensor))]
    for start in range(0, len(order), 32):
        batch = order[start:start + 32]
        # 2. The call
        logits = net(image_tensor[batch], table_tensor[batch])
        loss = loss_function(logits, y_tensor[batch])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch) / len(order)
    if (epoch + 1) % 10 == 0:
        print("epoch", epoch + 1, "training loss", round(epoch_loss, 3))

net.eval()
with torch.no_grad():
    p_intermediate = torch.sigmoid(net(image_tensor[test_idx], table_tensor[test_idx])).numpy()
results["intermediate fusion"] = roc_auc_score(y[test_idx], p_intermediate)
print("test AUC:", results["intermediate fusion"])

**You should see** four lines of loss, and a test AUC of about 0.74 (the loss goes from about 0.60 to 0.54).

The loss goes down, but the test AUC is no better than a single source (0.76 and 0.74), and well below early fusion
with measurements (0.87). This is the result of one run: with 420 patients a network that must
learn the image numbers from scratch has too little data. The idea is good, the data are too few. With
thousands of patients, or with a pretrained image branch, the order could change.

# STEP 10: Compare, and open the box

The cell draws all the AUCs of `results`. Run it.

In [ ]:
names = sorted(results, key=results.get)
plt.figure(figsize=(8, 4))
plt.barh(names, [results[n] for n in names])
plt.axvline(0.5, color="gray", linestyle="--")
plt.xlabel("test AUC")
plt.xlim(0.4, 1)
plt.show()

Now we open the box. The next cell prints the formula that generated the clinical table and the outcome. Run
it and read it.

In [ ]:
import inspect

print(inspect.getsource(make_clinical_table))

Answer in words, in a text cell:

1. Which variables really changed the risk of dying?
2. Which model had access to which of them? Look at the bar chart again.
3. Why did `sex` not help?

**Careful.**

- The data are invented. On real data fusion often helps less, because real clinical data are noisier and
  the image and the table often tell the same thing.
- The test set has 180 patients. Differences of 0.02 or 0.03 between two models are noise. Change
  `random_state` in the split of STEP 3, run everything again, and see how the ranking moves.
- "Alive at one year" as yes or no ignores the patients we lost before one year, and the time of death. Real
  studies use survival analysis.

**Answers**

1. The size of the nodule (the strongest), the performance status, the age, the smoking history and the mean
   density of the nodule. The sex has no effect.
2. The clinical model saw age, smoking and performance status, but not the size and the density. The
   measurements model saw size and density only. Early and late fusion saw both. The embeddings carry the
   size only in a vague way.
3. Because the formula does not contain it. A model can only find what is in the data. It was given the
   column and learned a weight close to zero.

# If you have finished

1. Use a random forest (`RandomForestClassifier`) instead of the logistic regression in `test_auc`. Does the ranking change?
2. Add 20 columns of random noise to `X_early`. What happens to the AUC?
3. In late fusion, use weights instead of the plain average (for example 0.3 and 0.7). Which weights are best?
4. Tell the network where to look. Cut a 32 x 32 crop around the centre of the mask of each nodule
   (hint: `np.argwhere(mask).mean(axis=0)`, keep the crop inside the patch). Compute the embeddings of the crops
   as in STEP 8, reduce them with a PCA, and test them alone and with `X_clinical`. Is it better than the whole patch?
5. Another network: DINOv2. With the library `transformers`, load `AutoModel.from_pretrained("facebook/dinov2-small")`,
   prepare the input as in STEP 8, and keep the first token of `last_hidden_state` as the embedding (384 numbers).
   Test it on the whole patch and on the crop.

The handbook, chapter 5, explains what you are comparing.

**Solutions** (the code is run below)

1. In `test_auc`, replace the model with `RandomForestClassifier(n_estimators=300, random_state=0)`.
2. Add `rng.normal(size=(600, 20))` to `X_early` with `np.concatenate`.
3. `p_late = w * p_clinical + (1 - w) * p_measurements` for a few values of `w`. Choosing the weight on the
   test patients is cheating: it should be chosen on a validation set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_early.values[train_idx], y[train_idx])
print("1. random forest, early fusion:", roc_auc_score(y[test_idx], forest.predict_proba(X_early.values[test_idx])[:, 1]))

noise = np.random.default_rng(0).normal(size=(len(y), 20))
print("2. early fusion with 20 noise columns:", test_auc(np.concatenate([X_early.values, noise], axis=1))[0])

for w in [0.3, 0.5, 0.7]:
    print("3. late fusion, weight of the clinical model", w, ":", roc_auc_score(y[test_idx], w * p_clinical + (1 - w) * p_measurements))

**Solutions of 4 and 5** (the code is run below). The helper `embed` repeats the preparation of STEP 8 for any
set of patches and any network. The measured AUCs are in the comments.

In [ ]:
!pip install -q transformers
from transformers import AutoModel

# 4. A crop of 32 x 32 around the centre of the mask, kept inside the patch of 64 x 64
crops = []
for image, mask in zip(images, masks):
    centre_x, centre_y = [int(round(v)) for v in np.argwhere(mask).mean(axis=0)]
    centre_x = min(max(centre_x, 16), 48)
    centre_y = min(max(centre_y, 16), 48)
    crops.append(image[centre_x - 16:centre_x + 16, centre_y - 16:centre_y + 16])
crops = np.stack(crops)


def embed(patches, get_features):
    # the preparation of STEP 8: window, three channels, 224 x 224, mean and std of ImageNet
    x = torch.tensor(np.clip((patches + 1000) / 1400, 0, 1), dtype=torch.float32)[:, None]
    x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    x = (x - torch.tensor([0.485, 0.456, 0.406])[None, :, None, None]) / torch.tensor([0.229, 0.224, 0.225])[None, :, None, None]
    with torch.no_grad():
        return torch.cat([get_features(x[start:start + 50]) for start in range(0, len(x), 50)]).numpy()


def test_embeddings(embedding_table, label):
    # PCA on the training patients only, then the AUC alone and with the clinical data
    pca = PCA(n_components=10, random_state=0).fit(embedding_table[train_idx])
    reduced = pca.transform(embedding_table)
    alone, _ = test_auc(reduced)
    with_clinical, _ = test_auc(np.concatenate([reduced, X_clinical.values], axis=1))
    print(f"{label}: alone {alone:.3f}, with the clinical data {with_clinical:.3f}")


densenet = DenseNet121(spatial_dims=2, in_channels=3, out_channels=1000, pretrained=True).eval()
densenet_features = lambda batch: F.adaptive_avg_pool2d(F.relu(densenet.features(batch)), 1).flatten(1)

test_embeddings(embed(images, densenet_features), "4. DenseNet, whole patch")   # 0.62 and 0.79, as in STEP 8
test_embeddings(embed(crops, densenet_features), "4. DenseNet, crop")           # about 0.78 and 0.87

# 5. DINOv2: the first token of the last layer
dino = AutoModel.from_pretrained("facebook/dinov2-small").eval()
dino_features = lambda batch: dino(pixel_values=batch).last_hidden_state[:, 0]

test_embeddings(embed(images, dino_features), "5. DINOv2, whole patch")   # about 0.57 and 0.75
test_embeddings(embed(crops, dino_features), "5. DINOv2, crop")           # about 0.67 and 0.81

The crop helps both networks a lot: the network looks at the nodule and not at the lung around it. DenseNet goes
from 0.62 to 0.78 alone and from 0.79 to 0.87 with the clinical data; DINOv2 from 0.57 to 0.67 and from 0.75 to 0.81.
The newer network is not better: DINOv2 is below DenseNet in every row. With the crop, DenseNet does as well as the
five measurements of STEP 5 (0.74 alone, 0.87 with the clinical data). Other results of the solutions: the random
forest on early fusion reaches 0.83, not better than the logistic regression (0.87); 20 noise columns cost about
0.03 (0.87 to 0.83); for the weight of the clinical model in late fusion, 0.3 gives 0.84, 0.5 gives 0.86 and 0.7
gives 0.83, so the plain average is the best of the three.

# Check yourself

1. What is the difference between early, intermediate and late fusion?
2. Why is the PCA fitted on the training patients only?
3. Early fusion has an AUC of 0.87 and late fusion 0.86. Is early fusion better?
4. A classmate says: "fusion always improves the prediction". What do you answer?

**Answers**

1. Early: the data are joined before the model, one model. Intermediate: one network, each source has its own
   branch, and the numbers of the branches are joined inside the network. Late: one model for each source,
   and the answers are combined at the end.
2. The test patients must not influence any step of the training. A PCA fitted on all patients has already
   seen the test images.
3. No. With 180 test patients a difference of 0.01 is noise. Repeat with other splits before saying anything.
4. It helps only if the second source holds information that the first does not. In this notebook it helped
   because the outcome was built from both. The embeddings added to the clinical data helped less than the
   measurements (0.79 against 0.87), and 20 columns of noise made early fusion worse.

# To go further

- Huang et al. 2020, fusion of medical imaging and electronic health records (early, joint, late fusion): https://pmc.ncbi.nlm.nih.gov/articles/PMC7567861
- MONAI tutorial, multimodal early fusion: https://github.com/Project-MONAI/tutorials/tree/main/multimodal
- CMU course on multimodal machine learning: https://cmu-mmml.github.io/
- PyRadiomics, the standard library for radiomic measurements: https://pyradiomics.readthedocs.io
- Survival analysis, the basics (Clark et al. 2003): https://pmc.ncbi.nlm.nih.gov/articles/PMC2394262/